# Descriptive forecast monitoring: controlled synthetic demonstration

IT4063E. No Rossmann source reads, LightGBM execution, production observations or automatic retraining. This replaces the earlier Evidently notebook whose target-derived pseudo-predictions were invalid. All calculation logic lives in `src/rossmann_forecasting/monitoring.py`.


In [ ]:
from datetime import UTC, datetime

import matplotlib.pyplot as plt
import pandas as pd

from rossmann_forecasting.monitoring import build_monitoring_summary, synthetic_monitoring_inputs

forecasts, labels = synthetic_monitoring_inputs()
summary = build_monitoring_summary(
    forecasts,
    labels,
    as_of="2015-07-03",
    reference_start="2015-06-06",
    reference_end="2015-06-19",
    current_start="2015-06-20",
    current_end="2015-07-03",
    generated_at=datetime.now(UTC).isoformat(),
)
print(summary["status"], summary["generated_at"], summary["evidence_kind"])

## Information boundary and populations

Predictions depend only on Store/calendar. Labels are authored afterward; the adapter validates issuance before pairing. A genuine caller must separately authenticate the immutable forecast artifact and obtain data access. Primary errors use observed Open=1 with available raw forecasts. MAPE excludes zero actuals; WAPE is a ratio. Missing labels/forecasts stay unavailable. Receipt hashes do not prove historical custody.


In [ ]:
pd.DataFrame(
    [summary["reference"]["pooled"], summary["current"]["pooled"]], index=["reference", "current"]
).T

In [ ]:
pd.DataFrame(summary["current"]["by_store"])

In [ ]:
pd.DataFrame(summary["current"]["by_horizon"])

## Calendar windows

The trailing window includes seven calendar days, including closures and missing-label days. Startup windows are partial. Forecast origin/horizon and observed-label counts are retained; rows are dependent forecast-label pairs, not independent experimental replicates.


In [ ]:
rolling = [row for row in summary["windows"] if row["nominal_days"] == 7]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot([row["end"] for row in rolling], [row["pooled"]["mae"] for row in rolling], marker="o")
ax.set(
    title="Synthetic trailing 7-calendar-day MAE; startup windows partial",
    ylabel="Monetary turnover units",
    xlabel="Window end",
)
ax.tick_params(axis="x", labelrotation=45)
fig.tight_layout()
plt.show()

## Descriptive characteristics and human review

Reference/current windows are disjoint. Features are declared origin-known calendar/planned values. Quartiles, frequency changes, sample counts and missingness have no KS significance claim or drift threshold. Constant references and insufficient samples remain explicit. No persistence trigger is retained. Evidence state is not system health; no real post-deployment performance can yet be assessed.


In [ ]:
pd.DataFrame(summary["feature_comparison"])

In [ ]:
print(summary["human_review"])
for limitation in summary["limitations"]:
    print("-", limitation)

## Reproduction and pending work

`uv run --locked python scripts/generate_monitoring_artifacts.py --output-dir reports/generated/monitoring/<new-run-id>` writes JSON, HTML, chart and provenance manifest from the same API. Genuine authorized development artifacts, final Phase 13 M5 integration, broader trigger approval/rehearsal and production labels remain pending. Team members must inspect, understand and accept AI-assisted code and claims.
